# Round 3 — the tuplet-mark A/B (`strips_v5_tupnew` vs `strips_v5_tupctl`)

**One experiment, two arms, run this notebook TWICE** — once with `ARM = 'tupnew'`, once with `ARM = 'tupctl'`. Everything else is held fixed: same pieces, same `data/split_v4.json`, same recipe, same seed. The corpora differ in exactly one thing, the shape of the triplet mark:

| arm | mark |
|---|---|
| `tupnew` | the measured shape — arc **broken**, the "3" set in the gap (16/16 real marks, ~11 editions) |
| `tupctl` | the **control** — the pre-2026-08-12 continuous arc with the digit floating above it |

The question: `\tup3` recall is **83.8%**, under its ≥85% floor and *below* its own pre-slur-distractor baseline of 92.7%. Does drawing the mark the way real print draws it recover that? Nothing about recall has ever been claimed for the redraw — this is the measurement.

**Read `docs/rung3/round3-criteria.md` before touching the result.** It pre-registers the one selection number (free-running `\tup3` recall on `_tupletval`), the guards, the McNemar test, and the decision rule — including what a null means. The pool is small: **54 gold groups**, so one group is 1.9 pp and nothing under ~11 pp can reach significance. That is written down on purpose, in advance.

⚠ **Exam strips are not on this VM and the exam is not read here.** It is read once, later, on Round 3's final model.

⚠ **The selection read happens on the Mac**, not here — `_tupletval` is 28 crops and a CPU eval, and keeping it local means the two arms are scored by one command with one code version.

In [ ]:
# ===== THE ONLY KNOB IN THIS NOTEBOOK =====
# Set it once, run the notebook through, then come back, change it, and run it again.
ARM = 'tupnew'   # 'tupnew' | 'tupctl'
assert ARM in ('tupnew', 'tupctl')
STRIPS = f'data/synthetic/strips_v5_{ARM}'
ZIP = f'tnc_round3_{ARM}_colab.zip'
DRIVE = '/content/drive/MyDrive/tnc'
print(ARM, STRIPS, ZIP)

In [ ]:
# Which GPU did we get? (T4 16GB / L4 24GB / A100 40GB)
!nvidia-smi

In [ ]:
# Mount Google Drive (approve the popup).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%time
# Copy this arm's package Drive -> VM disk and unzip (fast local disk for the dataloader).
!cp {DRIVE}/{ZIP} /content/
!rm -rf /content/tnc && mkdir /content/tnc
!cd /content/tnc && unzip -q /content/{ZIP}

# WHICH ARM IS ACTUALLY ON DISK. The two corpora have byte-identical manifests — the mark is
# pixels only — so this file is the ONLY thing that can tell them apart. A mixed-up upload would
# otherwise train the wrong arm and never say so.
import json
cfg = json.load(open(f'/content/tnc/{STRIPS}/render_config.json'))
print(cfg)
assert cfg['legacyTupletMark'] is (ARM == 'tupctl'), f"{ZIP} does not contain arm {ARM}"
assert cfg['thinSharps'] is True and cfg['printNoise'] is False
!wc -l /content/tnc/{STRIPS}/manifest.jsonl
!python -c "import json;s=json.load(open('/content/tnc/data/split_v4.json'));print('train',len(s['train_pieces']),'val',len(s['val_pieces']))"

In [ ]:
# Dependencies (torch + torchvision are preinstalled on Colab).
!pip -q install transformers albumentations opencv-python-headless

In [ ]:
# SHAKEOUT (~3 min): 150 tiny steps from BASE — a WIRING smoke, not a result.
# Expect: `vocab: +25 tokens -> 100 ids`, the three real pools listed, `exam-disjointness OK`,
# and val loss FALLING.
%cd /content/tnc
!python src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir data/real/rung3/strips_nota --real-dir data/real/rung3/strips_r1 \
    --real-dir data/real/rung3/strips_tup \
    --every-share 0.15 --out-dir /content/r3-shakeout \
    --lr 3e-5 --warmup-steps 30 --max-steps 150 --batch-size 8 \
    --limit-val 40 --eval-every 50 --save-every 50 --log-every 25 --num-workers 2

In [ ]:
# ===== CALIBRATE THROUGHPUT ON *THIS* RUNTIME (~2-3 min) — before any long run =====
#   hours = (steps * batch) / samples_per_sec / 3600
# Rung-2 reference: 6000 steps at batch 16 took ~110 min. Round-1 T4/2-vCPU was ~2.4 s/step at
# batch 8 and was AUGMENTATION-CPU-bound, not GPU-bound.
#
# ⚠ A/B DISCIPLINE: whatever you set here, set the SAME for both arms. --num-workers and the
# runtime type do not change the result, but --batch-size and step counts do.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!nproc   # vCPU count -> the ceiling on --num-workers
%cd /content/tnc
!python src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir data/real/rung3/strips_nota --real-dir data/real/rung3/strips_r1 \
    --real-dir data/real/rung3/strips_tup \
    --every-share 0.15 --out-dir /content/calib \
    --lr 3e-5 --warmup-steps 20 --max-steps 60 --batch-size 16 \
    --limit-val 8 --eval-every 60 --save-every 60 --log-every 20 --num-workers 10

In [ ]:
# ===== STAGE 1 — carry-dominant SYNTHETIC ONLY, from BASE =====
# No --real-dir: this builds the carry-native synthetic checkpoint stage 2 specialises. This is
# where the two arms actually differ — it is the only stage that sees the tuplet mark in bulk.
# Recipe identical to Round 1's winning Arm A and to Round 2, except for the corpus.
%cd /content/tnc
!python src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage1 \
    --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10  # ~nproc-2; T4 (2 vCPU) use 2

In [ ]:
# ===== STAGE 2 — real-SPECIALISATION fine-tune from stage 1 =====
# Fresh LOW lr + short warmup from the stage-1 checkpoint.
#
# `:9` as in Round 2 — the suffix oversamples each real pool so real is ~1/3 of batches. Re-check
# it if the corpus size moved: with ~36k synthetic train strips and ~2.1k real, :8 gives ~31.9%
# and :9 ~34.6% (Round 1 sat at 33.7%). Whatever it is, it must be THE SAME IN BOTH ARMS.
#
# Selection caveat carried over: oversampled real overfits fast and `best` is picked on a
# synth-dominated val mix — so both `best` and `last` come home for the paired read.
%cd /content/tnc
!python src/vision/train.py --model {DRIVE}/r3-{ARM}-stage1/best \
    --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir data/real/rung3/strips_nota:9 \
    --real-dir data/real/rung3/strips_r1:9 \
    --real-dir data/real/rung3/strips_tup:9 \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage2 \
    --lr 1e-5 --warmup-steps 100 --max-steps 2000 --batch-size 16 --num-workers 10

In [ ]:
# RESUME after a disconnect: re-run the setup cells, then this with the SAME flags as the stage
# you were running (edit out-dir/flags to match). --resume reloads model+optimizer+scheduler from
# <out-dir>/last and ignores --model.
%cd /content/tnc
!python src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage1 \
    --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10 --resume

In [ ]:
# ===== SANITY ONLY — did anything break? =====
# NOT the selection number. The A/B is decided on the Mac, on `_tupletval`, paired. This cell
# exists so a broken run is caught before it is downloaded, and to pick between `best` and `last`.
%cd /content/tnc
!python src/vision/make_realval_pool.py --real-dir data/real/rung3/strips_nota \
    --real-dir data/real/rung3/strips_r1 --real-dir data/real/rung3/strips_tup \
    --split data/split_v4.json

for ck in [f'r3-{ARM}-stage2/best', f'r3-{ARM}-stage2/last']:
    print('=' * 70, '\n==', ck)
    !python src/vision/eval_omr.py --checkpoint {DRIVE}/{ck} \
        --strips-dir data/real/rung3/_realval --split none --show-errors 0

## After BOTH arms have run

1. **Download both stage-2 checkpoints** from `MyDrive/tnc/r3-tupnew-stage2/` and `r3-tupctl-stage2/` into `data/checkpoints/` on the Mac.
2. **Read the pre-registered number and its paired test — one command:**
   ```bash
   .venv-ml/bin/python scripts/rung3/tuplet_ab_score.py \
       --new data/checkpoints/<tupnew-ckpt> --ctl data/checkpoints/<tupctl-ckpt>
   ```
   It decodes `_tupletval` with both arms, scores every gold `\tup3` occurrence by the same alignment `eval_omr.py` uses, and prints each arm's recall/precision plus the **exact McNemar p** on the discordant groups. With 54 groups, ~6 discordant groups all one way is the threshold of significance; a smaller difference is a **null**, and a null is reported as a null.
3. **Read the guard** on `_realval_v2` — `eval_omr.py --strips-dir data/real/rung3/_realval_v2 --split none`; mean AEU F1 must not fall more than 1 pp between arms, and `\tup3` precision ≥70% is a veto, not a tiebreak.
4. **Apply the decision rule in `docs/rung3/round3-criteria.md`** — it is written down; do not re-derive it from the result. In particular: a null **keeps** the redraw (it is measured against real print) and claims nothing.
5. **The exam is still unread.** It is one shot, on Round 3's final model, against the floors in the same file.
6. **Attribution note to write down:** both arms were rendered from `data/pieces_v4.json` with `--thin-sharps` and no print noise, so within the A/B only the mark moved. Against **Round 2** the corpus also differs by the 2026-08-05 unclosed-`\tup3` rhythm fix, which touches 5 measures in 1 piece.